# Сегментация дорожных знаков (YOLO11n-seg) + трекинг (ByteTrack, BoT-SORT)


In [1]:
%pip -q install ultralytics kaggle opencv-python pyyaml tqdm motmetrics lap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.8 MB/s eta 0:00:00


In [2]:
import sys

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import json
import csv
import random
import pandas as pd
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

torch.backends.cudnn.benchmark = True

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
SEED = 555
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def get_device():
    if torch.cuda.is_available():
        return 0
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
DEVICE

0

In [5]:
def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def save_yaml(path: Path, data) -> None:
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

def try_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

def kaggle_auth() -> bool:
    if os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"):
        return True
    u = try_colab_secret("KAGGLE_USERNAME")
    k = try_colab_secret("KAGGLE_KEY")
    if u and k:
        os.environ["KAGGLE_USERNAME"] = u
        os.environ["KAGGLE_KEY"] = k
        return True
    return (Path.home() / ".kaggle" / "kaggle.json").exists()

def kaggle_download(dataset: str, out_dir: Path) -> Path:
    out_dir = ensure_dir(out_dir)
    marker = out_dir / ".kaggle_done"
    if marker.exists():
        return out_dir
    if not kaggle_auth():
        raise RuntimeError("no kaggle credentials")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", dataset, "-p", str(out_dir), "--unzip"],
        check=True,
    )
    marker.write_text("ok", encoding="utf-8")
    return out_dir

In [6]:
ROOT = ensure_dir(Path("/content/drive/MyDrive/runs_road_signs_seg")) if IS_COLAB else ensure_dir(Path("./runs_road_signs_seg").resolve())
WORK = ensure_dir(Path("/content/work_road_signs_seg")) if IS_COLAB else ensure_dir((ROOT / "work").resolve())

YOLO_READY_DIR = Path('/content/drive/MyDrive/computer_vision/yolo_dataset_lab3/sign_dataset_yolo_seg')
RAW_DIR = ensure_dir(WORK / "kaggle_dataset")

VIDEOS_DIR = ensure_dir(ROOT / "videos")
GT_TRACKS_DIR = ROOT / "gt_tracks"
RESULTS_DIR = ensure_dir(ROOT / "results")

KAGGLE_DATASET = "viacheslavshalamov/russian-road-signs-segmentation-dataset"

IMGSZ = 640
BATCH = 32
EPOCHS = 60
CONF_THRES = 0.35
MASK_THRES = 0.5

print("ROOT:", ROOT)
print("YOLO_READY_DIR:", YOLO_READY_DIR)
print("VIDEOS_DIR:", VIDEOS_DIR)

ROOT: /content/drive/MyDrive/runs_road_signs_seg
YOLO_READY_DIR: /content/drive/MyDrive/computer_vision/yolo_dataset_lab3/sign_dataset_yolo_seg
VIDEOS_DIR: /content/drive/MyDrive/runs_road_signs_seg/videos


In [7]:
USE_READY_YOLO = (YOLO_READY_DIR / "data.yaml").exists()

if USE_READY_YOLO:
    DATA_YAML = YOLO_READY_DIR / "data.yaml"
    print("Using prepared YOLO dataset:", DATA_YAML)
else:
    kaggle_download(KAGGLE_DATASET, RAW_DIR)

    def _pick_content_dir(base: Path) -> Path:
        best = None
        best_score = (-1, -1)
        for d in [base] + [p for p in base.iterdir() if p.is_dir()]:
            jpg_cnt = sum(1 for _ in d.rglob("*.jpg"))
            json_cnt = sum(1 for _ in d.rglob("*.json"))
            if jpg_cnt and json_cnt and (jpg_cnt, json_cnt) > best_score:
                best = d
                best_score = (jpg_cnt, json_cnt)
        return best or base

    EXTRACT_DIR = _pick_content_dir(RAW_DIR)
    DATA_YAML = YOLO_READY_DIR / "data.yaml"
    print("Prepared YOLO dataset not found. Raw source:", EXTRACT_DIR)
    print("Will build YOLO dataset into:", YOLO_READY_DIR)

Using prepared YOLO dataset: /content/drive/MyDrive/computer_vision/yolo_dataset_lab3/sign_dataset_yolo_seg/data.yaml


In [8]:
if USE_READY_YOLO:
    print("Skip conversion")
else:
    names = [
        "warning",
        "priority",
        "prohibitory",
        "mandatory",
        "special_regulations",
        "information",
        "service",
        "additional_information",
    ]

    def _find_dir(root: Path, name: str) -> Optional[Path]:
        for d in root.rglob("*"):
            if d.is_dir() and d.name.lower() == name.lower():
                return d
        return None

    def _list_images(d: Path) -> List[Path]:
        exts = {".jpg", ".jpeg", ".png"}
        return sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in exts])

    def _ann_path(img_path: Path) -> Path:
        return img_path.parent / f"{img_path.name}_coco.json"

    def _pairs(d: Path) -> List[Tuple[Path, Path]]:
        out = []
        for img in _list_images(d):
            ann = _ann_path(img)
            if ann.exists():
                out.append((img, ann))
        return out

    def _load_json(path: Path) -> dict:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, dict) and "root" in obj and isinstance(obj["root"], dict):
            obj = obj["root"]
        return obj

    def _superclass(raw_cls) -> Optional[int]:
        try:
            v = int(raw_cls)
        except Exception:
            return None
        if 1 <= v <= 8:
            return v - 1
        if 0 <= v <= 7:
            return v
        return None

    def _safe_link_or_copy(src: Path, dst: Path) -> None:
        if dst.exists():
            return
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            dst.symlink_to(src)
        except Exception:
            import shutil
            shutil.copy2(src, dst)

    def _mask_to_polys(mask: np.ndarray) -> List[np.ndarray]:
        mask = (mask > 0).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        polys = []
        for cnt in cnts:
            if cnt.shape[0] < 3 or cv2.contourArea(cnt) < 5.0:
                continue
            eps = 0.002 * cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, eps, True)
            if approx.shape[0] >= 3:
                polys.append(approx.reshape(-1, 2).astype(np.float32))
        return polys

    def _roi_to_img(poly: np.ndarray, box: List[float], roi_w: int, roi_h: int) -> np.ndarray:
        x1, y1, x2, y2 = map(float, box[:4])
        sx = (x2 - x1) / max(1.0, float(roi_w - 1))
        sy = (y2 - y1) / max(1.0, float(roi_h - 1))
        out = poly.copy()
        out[:, 0] = x1 + out[:, 0] * sx
        out[:, 1] = y1 + out[:, 1] * sy
        return out

    def _norm_poly(poly: np.ndarray, w: int, h: int) -> np.ndarray:
        out = poly.copy()
        out[:, 0] = np.clip(out[:, 0] / max(1, w), 0.0, 1.0)
        out[:, 1] = np.clip(out[:, 1] / max(1, h), 0.0, 1.0)
        return out

    def _json_to_lines(img_path: Path, ann_path: Path) -> List[str]:
        img = cv2.imread(str(img_path))
        if img is None:
            return []
        h, w = img.shape[:2]
        ann = _load_json(ann_path)

        masks = ann.get("masks") or []
        boxes = ann.get("bbox") or ann.get("rois") or []
        class_ids = ann.get("class_ids") or []

        n = min(len(masks), len(boxes), len(class_ids))
        lines = []

        for i in range(n):
            cls = _superclass(class_ids[i])
            if cls is None:
                continue

            patch = np.asarray(masks[i], dtype=np.uint8)
            if patch.ndim != 2 or patch.size == 0:
                continue

            roi_h, roi_w = patch.shape[:2]
            for poly in _mask_to_polys(patch):
                poly = _roi_to_img(poly, boxes[i], roi_w, roi_h)
                poly = _norm_poly(poly, w, h)
                flat = poly.reshape(-1).tolist()
                if len(flat) >= 6:
                    lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat) + "\n")
        return lines

    train_dir = _find_dir(EXTRACT_DIR, "train")
    val_dir = _find_dir(EXTRACT_DIR, "val")
    assert train_dir is not None and val_dir is not None, "Не найдены train/val"

    train_pairs = _pairs(train_dir)
    val_pairs = _pairs(val_dir)

    print("train pairs:", len(train_pairs))
    print("val pairs  :", len(val_pairs))

    train_img_dir = ensure_dir(YOLO_READY_DIR / "train" / "images")
    train_lbl_dir = ensure_dir(YOLO_READY_DIR / "train" / "labels")
    val_img_dir = ensure_dir(YOLO_READY_DIR / "val" / "images")
    val_lbl_dir = ensure_dir(YOLO_READY_DIR / "val" / "labels")

    for img_path, ann_path in tqdm(train_pairs, desc="prepare_train"):
        dst_img = train_img_dir / img_path.name
        dst_lbl = train_lbl_dir / f"{img_path.stem}.txt"
        _safe_link_or_copy(img_path, dst_img)
        dst_lbl.write_text("".join(_json_to_lines(img_path, ann_path)), encoding="utf-8")

    for img_path, ann_path in tqdm(val_pairs, desc="prepare_val"):
        dst_img = val_img_dir / img_path.name
        dst_lbl = val_lbl_dir / f"{img_path.stem}.txt"
        _safe_link_or_copy(img_path, dst_img)
        dst_lbl.write_text("".join(_json_to_lines(img_path, ann_path)), encoding="utf-8")

    save_yaml(
        DATA_YAML,
        {
            "path": str(YOLO_READY_DIR),
            "train": "train/images",
            "val": "val/images",
            "nc": 8,
            "names": names,
        },
    )
    print("Built:", DATA_YAML)

Skip conversion


In [9]:
WEIGHTS = "yolo11n-seg.pt"
_ = YOLO(WEIGHTS)
WEIGHTS

'yolo11n-seg.pt'

In [10]:
def train_segmenter(weights: str, data_yaml: Path, name: str):
    save_dir = ROOT / name
    best_existing = save_dir / "weights" / "best.pt"
    if best_existing.exists():
        print("Using existing model:", best_existing)
        return save_dir, best_existing

    model = YOLO(weights)
    result = model.train(
        data=str(data_yaml),
        task="segment",
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        workers=2,
        cache=False,
        patience=20,
        seed=SEED,
        project=str(ROOT),
        name=name,
        exist_ok=True,
        verbose=True,
    )
    best = Path(result.save_dir) / "weights" / "best.pt"
    return Path(result.save_dir), best

run_dir, best_path = train_segmenter(WEIGHTS, DATA_YAML, "yolo_seg_8_signs")
print("best:", best_path)

best_model = YOLO(str(best_path))

Using existing model: /content/drive/MyDrive/runs_road_signs_seg/yolo_seg_8_signs/weights/best.pt
best: /content/drive/MyDrive/runs_road_signs_seg/yolo_seg_8_signs/weights/best.pt


## Метрики сегментации на валидации

In [11]:
def _resolve_val_images(data_yaml: Path):
    data = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    val_ref = Path(str(data["val"]))
    exts = {".jpg", ".jpeg", ".png"}

    cleaned = Path(*[p for p in val_ref.parts if p not in {".", ".."}])
    val_dir = (data_yaml.parent / cleaned).resolve()

    imgs = sorted([x for x in val_dir.rglob("*") if x.is_file() and x.suffix.lower() in exts])

    print("DATA_YAML:", data_yaml)
    print("val ref  :", data["val"])
    print("val dir  :", val_dir)
    print("images   :", len(imgs))

    if not imgs:
        raise FileNotFoundError(f"Validation images not found: {val_dir}")

    return imgs

def _img_to_label(img_path: Path) -> Path:
    parts = list(img_path.parts)
    if "images" in parts:
        parts[parts.index("images")] = "labels"
        return Path(*parts).with_suffix(".txt")
    return img_path.with_suffix(".txt")

def _union_gt_mask(lbl_path: Path, w: int, h: int) -> np.ndarray:
    m = np.zeros((h, w), dtype=np.uint8)
    if not lbl_path.exists():
        return m

    for ln in lbl_path.read_text(encoding="utf-8").splitlines():
        vals = ln.strip().split()
        if len(vals) < 7:
            continue
        coords = np.asarray([float(x) for x in vals[1:]], dtype=np.float32)
        if coords.size % 2 != 0:
            continue
        pts = coords.reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        pts = pts.astype(np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(m, [pts], 1)
    return m

def _union_pred_mask(res, w: int, h: int) -> np.ndarray:
    m = np.zeros((h, w), dtype=np.uint8)
    if res.masks is None:
        return m
    for x in res.masks.data.detach().cpu().numpy():
        bm = (x >= MASK_THRES).astype(np.uint8)
        if bm.shape[:2] != (h, w):
            bm = cv2.resize(bm, (w, h), interpolation=cv2.INTER_NEAREST)
        m = np.maximum(m, bm)
    return m

def _iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    return float(inter / union) if union > 0 else 0.0

def _l2(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)))

def evaluate_val_metrics(model: YOLO, data_yaml: Path):
    image_paths = _resolve_val_images(data_yaml)

    r = model.val(
        data=str(data_yaml),
        imgsz=IMGSZ,
        batch=min(BATCH, 8),
        conf=CONF_THRES,
        device=DEVICE,
        split="val",
        plots=False,
        verbose=False,
    )
    d = dict(getattr(r, "results_dict", {}) or {})

    ious, l2s = [], []
    for img_path in tqdm(image_paths, desc="eval_val"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        gt = _union_gt_mask(_img_to_label(img_path), w, h)
        pred = model.predict(
            source=str(img_path),
            imgsz=IMGSZ,
            conf=CONF_THRES,
            device=DEVICE,
            verbose=False,
        )[0]
        pr = _union_pred_mask(pred, w, h)

        ious.append(_iou(gt, pr))
        l2s.append(_l2(gt, pr))

    metrics = {
        "box_precision": float(d.get("metrics/precision(B)", 0.0)),
        "box_recall": float(d.get("metrics/recall(B)", 0.0)),
        "box_mAP50": float(d.get("metrics/mAP50(B)", 0.0)),
        "box_mAP50_95": float(d.get("metrics/mAP50-95(B)", 0.0)),
        "mask_precision": float(d.get("metrics/precision(M)", 0.0)),
        "mask_recall": float(d.get("metrics/recall(M)", 0.0)),
        "mask_mAP50": float(d.get("metrics/mAP50(M)", 0.0)),
        "mask_mAP50_95": float(d.get("metrics/mAP50-95(M)", 0.0)),
        "IoU_mean": float(np.mean(ious)) if ious else 0.0,
        "L2_mean": float(np.mean(l2s)) if l2s else 1.0,
        "pct_images_iou_ge_0_5": float(np.mean(np.asarray(ious) >= 0.5)) if ious else 0.0,
        "pct_images_iou_ge_0_75": float(np.mean(np.asarray(ious) >= 0.75)) if ious else 0.0,
        "pct_images_iou_ge_0_9": float(np.mean(np.asarray(ious) >= 0.9)) if ious else 0.0,
    }
    return metrics

val_metrics = evaluate_val_metrics(best_model, DATA_YAML)
for k, v in val_metrics.items():
    print(f"{k}: {v:.6f}")

DATA_YAML: /content/drive/MyDrive/computer_vision/yolo_dataset_lab3/sign_dataset_yolo_seg/data.yaml
val ref  : ../val/images
val dir  : /content/drive/MyDrive/computer_vision/yolo_dataset_lab3/sign_dataset_yolo_seg/val/images
images   : 127
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-seg summary (fused): 114 layers, 2,836,128 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 1.4±1.8 ms, read: 13.4±29.6 MB/s, size: 106.6 KB)
val: Scanning /content/drive/MyDrive/computer_vision/yolo_dataset_lab3/sign_dataset_yolo_seg/val/labels.cache... 127 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 127/127 26.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.6s/it 24.8s
                   all        127        327      0.465      0.367      0.437      0.347      0.458      0.361      0.428      0.271
Speed: 1.1

eval_val:   0%|          | 0/127 [00:00<?, ?it/s]

box_precision: 0.464977
box_recall: 0.366973
box_mAP50: 0.436563
box_mAP50_95: 0.346924
mask_precision: 0.458164
mask_recall: 0.360617
mask_mAP50: 0.428047
mask_mAP50_95: 0.270853
IoU_mean: 0.577485
L2_mean: 0.054857
pct_images_iou_ge_0_5: 0.803150
pct_images_iou_ge_0_75: 0.110236
pct_images_iou_ge_0_9: 0.007874


## Трекинг на 3+ видео

In [12]:
def list_mp4(d: Path):
    return sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() == ".mp4"])


videos = list_mp4(VIDEOS_DIR)
print("videos:", [p.name for p in videos])


def track_video_to_mot(model: YOLO, video_path: Path, tracker: str, out_dir: Path):
    tracker_name = Path(tracker).stem
    run_name = f"{video_path.stem}_{tracker_name}"
    save_dir = ensure_dir(out_dir / run_name)
    out_mot = save_dir / f"{video_path.stem}_{tracker_name}.txt"

    saved_videos = sorted(save_dir.glob("*.mp4")) + sorted(save_dir.glob("*.avi"))
    if out_mot.exists() and saved_videos:
        return saved_videos[0], out_mot

    with out_mot.open("w", encoding="utf-8") as f:
        for frame_i, res in enumerate(
            model.track(
                source=str(video_path),
                stream=True,
                tracker=tracker,
                conf=CONF_THRES,
                imgsz=IMGSZ,
                device=DEVICE,
                persist=True,
                save=True,
                project=str(out_dir),
                name=run_name,
                exist_ok=True,
                verbose=False,
            ),
            start=1,
        ):
            if res.boxes is None or res.boxes.id is None:
                continue

            ids = res.boxes.id.detach().cpu().numpy().astype(int)
            xyxy = res.boxes.xyxy.detach().cpu().numpy().astype(float)
            conf = res.boxes.conf.detach().cpu().numpy().astype(float)
            cls = res.boxes.cls.detach().cpu().numpy().astype(int)

            for tid, bb, sc, c in zip(ids, xyxy, conf, cls):
                x1, y1, x2, y2 = bb.tolist()
                f.write(
                    f"{frame_i},{tid},{x1:.2f},{y1:.2f},{x2 - x1:.2f},{y2 - y1:.2f},{sc:.4f},{c},-1\n"
                )

    saved_videos = sorted(save_dir.glob("*.mp4")) + sorted(save_dir.glob("*.avi"))
    out_video = saved_videos[0] if saved_videos else None
    return out_video, out_mot


out_dir = ensure_dir(RESULTS_DIR / "video_tracking")

for vp in videos:
    out_v1, pr1 = track_video_to_mot(best_model, vp, "bytetrack.yaml", out_dir)
    out_v2, pr2 = track_video_to_mot(best_model, vp, "botsort.yaml", out_dir)

    print("saved:", out_v1.name if out_v1 else "no_video", out_v2.name if out_v2 else "no_video")

videos: ['2026-03-08 17.42.19.mp4', '2026-03-08 17.47.54.mp4', '2026-03-08 17.48.02.mp4']
saved: 2026-03-08 17.42.19.avi 2026-03-08 17.42.19.avi
saved: 2026-03-08 17.47.54.avi 2026-03-08 17.47.54.avi
saved: 2026-03-08 17.48.02.avi 2026-03-08 17.48.02.avi


In [13]:
def read_mot(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        header=None,
        names=["frame", "id", "x", "y", "w", "h", "conf", "cls", "vis"],
    )
    df["frame"] = df["frame"].astype(int)
    df["id"] = df["id"].astype(int)
    return df

def simple_tracking_instability(mot_path: Path) -> dict:
    df = read_mot(mot_path)

    if df.empty:
        return {
            "file": mot_path.name,
            "frames": 0,
            "unique_ids": 0,
            "max_objects_one_frame": 0,
            "id_overgrowth_ratio": 0.0,
            "one_frame_ids": 0,
            "short_ids_le_2_frames": 0,
            "fragmented_ids": 0,
        }

    per_frame = df.groupby("frame")["id"].nunique()
    per_id = df.groupby("id")["frame"].apply(lambda s: sorted(set(s.tolist())))

    unique_ids = int(per_id.shape[0])
    max_objects_one_frame = int(per_frame.max()) if len(per_frame) else 0

    one_frame_ids = 0
    short_ids_le_2_frames = 0
    fragmented_ids = 0

    for frames in per_id:
        if len(frames) == 1:
            one_frame_ids += 1
        if len(frames) <= 2:
            short_ids_le_2_frames += 1
        if len(frames) >= 2 and np.any(np.diff(frames) > 1):
            fragmented_ids += 1

    return {
        "file": mot_path.name,
        "frames": int(df["frame"].nunique()),
        "unique_ids": unique_ids,
        "max_objects_one_frame": max_objects_one_frame,
        "id_overgrowth_ratio": unique_ids / max_objects_one_frame if max_objects_one_frame else 0.0,
        "one_frame_ids": one_frame_ids,
        "short_ids_le_2_frames": short_ids_le_2_frames,
        "fragmented_ids": fragmented_ids,
    }

track_dir = RESULTS_DIR / "video_tracking"

mot_files = sorted(
    p
    for subdir in track_dir.iterdir()
    if subdir.is_dir()
    for p in subdir.glob("*.txt")
    if p.name.endswith("_botsort.txt") or p.name.endswith("_bytetrack.txt")
)

print("track_dir:", track_dir)
print("found mot files:", len(mot_files))
for p in mot_files:
    print(p)

report = pd.DataFrame([simple_tracking_instability(p) for p in mot_files])
report = report.sort_values("file").reset_index(drop=True)
report

track_dir: /content/drive/MyDrive/runs_road_signs_seg/results/video_tracking
found mot files: 6
/content/drive/MyDrive/runs_road_signs_seg/results/video_tracking/2026-03-08 17.42.19_botsort/2026-03-08 17.42.19_botsort.txt
/content/drive/MyDrive/runs_road_signs_seg/results/video_tracking/2026-03-08 17.42.19_bytetrack/2026-03-08 17.42.19_bytetrack.txt
/content/drive/MyDrive/runs_road_signs_seg/results/video_tracking/2026-03-08 17.47.54_botsort/2026-03-08 17.47.54_botsort.txt
/content/drive/MyDrive/runs_road_signs_seg/results/video_tracking/2026-03-08 17.47.54_bytetrack/2026-03-08 17.47.54_bytetrack.txt
/content/drive/MyDrive/runs_road_signs_seg/results/video_tracking/2026-03-08 17.48.02_botsort/2026-03-08 17.48.02_botsort.txt
/content/drive/MyDrive/runs_road_signs_seg/results/video_tracking/2026-03-08 17.48.02_bytetrack/2026-03-08 17.48.02_bytetrack.txt


,file,frames,unique_ids,max_objects_one_frame,id_overgrowth_ratio,one_frame_ids,short_ids_le_2_frames,fragmented_ids
0,2026-03-08 17.42.19_botsort.txt,886,90,2,45.0,20,37,30
1,2026-03-08 17.42.19_bytetrack.txt,886,90,2,45.0,20,37,30
2,2026-03-08 17.47.54_botsort.txt,1213,194,4,48.5,39,66,81
3,2026-03-08 17.47.54_bytetrack.txt,1213,194,4,48.5,39,66,81
4,2026-03-08 17.48.02_botsort.txt,1293,165,5,33.0,31,47,51
5,2026-03-08 17.48.02_bytetrack.txt,1293,165,5,33.0,31,47,51


## Вывод
Трекеры ByteTrack и BoT-SORT показали одинаковые результаты на всех видео.
При небольшом числе объектов на кадре (2–5) было создано значительно больше уникальных ID (90–194), что говорит о нестабильности трекинга.
Большое количество коротких и фрагментированных треков указывает на частые потери объектов и повторное назначение новых ID.